## Practice Lab Exercise: Comparative Study of KNN Classification using the Iris Dataset

### Problem Statement
A botanical research team wants to classify iris flowers into different species based on petal and sepal measurements. You are required to build and analyse a K-Nearest Neighbours (KNN) classification model using both manual implementation and Scikit-learn. The study should investigate how the choice of K, distance metrics, feature scaling, cross-validation, and decision boundaries influence classification performance.

### Dataset Link:
[https://archive.ics.uci.edu/dataset/53/iris](https://archive.ics.uci.edu/dataset/53/iris)

## Setup and Installation

In [6]:
# Install and Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## Task 1: Dataset Understanding and Preprocessing

In [7]:
# Load Iris dataset
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name='target')

# Display first 10 records
print("First 10 records of Iris dataset:")
display(X.head(10))
print("\nTarget values for first 10 records:")
display(y.head(10))

# List input features and target classes
print("\nInput features:")
print(iris.feature_names)
print("\nTarget classes:")
print(iris.target_names)

# Check for missing values and data types
print("\nMissing values per column:")
print(X.isnull().sum())
print("\nData types:")
print(X.dtypes)

# Dataset statistics
print("\nDataset statistics:")
display(X.describe())

# Check class distribution
print("\nClass distribution:")
print(y.value_counts())
print(f"\nTotal samples: {len(X)}")

First 10 records of Iris dataset:


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2
5,5.4,3.9,1.7,0.4
6,4.6,3.4,1.4,0.3
7,5.0,3.4,1.5,0.2
8,4.4,2.9,1.4,0.2
9,4.9,3.1,1.5,0.1



Target values for first 10 records:


,target
0,0
1,0
2,0
3,0
4,0
5,0
6,0
7,0
8,0
9,0



Input features:
['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

Target classes:
['setosa' 'versicolor' 'virginica']

Missing values per column:
sepal length (cm)    0
sepal width (cm)     0
petal length (cm)    0
petal width (cm)     0
dtype: int64

Data types:
sepal length (cm)    float64
sepal width (cm)     float64
petal length (cm)    float64
petal width (cm)     float64
dtype: object

Dataset statistics:


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333
std,0.828066,0.435866,1.765298,0.762238
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000



Class distribution:
target
0    50
1    50
2    50
Name: count, dtype: int64

Total samples: 150


In [8]:
# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("Original data (first 5 rows):")
display(X.head())
print("\nScaled data (first 5 rows):")
display(X_scaled_df.head())

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)

print(f"\nTraining set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"\nTraining set class distribution:")
print(pd.Series(y_train).value_counts())
print(f"\nTest set class distribution:")
print(pd.Series(y_test).value_counts())

Original data (first 5 rows):


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2



Scaled data (first 5 rows):


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,-0.900681,1.019004,-1.340227,-1.315444
1,-1.143017,-0.131979,-1.340227,-1.315444
2,-1.385353,0.328414,-1.397064,-1.315444
3,-1.506521,0.098217,-1.283389,-1.315444
4,-1.021849,1.249201,-1.340227,-1.315444



Training set size: 105
Test set size: 45

Training set class distribution:
target
1    35
0    35
2    35
Name: count, dtype: int64

Test set class distribution:
target
2    15
1    15
0    15
Name: count, dtype: int64


### Explanation of why feature scaling is needed for KNN

KNN (K-Nearest Neighbors) relies on distance metrics (like Euclidean distance) to find the 'nearest' data points. When features have different scales or units, the features with larger numerical ranges will disproportionately influence the distance calculation. This can lead to biased results where the model prioritizes features with larger magnitudes, even if they are not more important for classification.

**Example:**
Imagine two features: 'sepal length (cm)' which ranges from ~4 to 8 cm, and 'petal width (cm)' which ranges from ~0.1 to 2.5 cm.

If we calculate the Euclidean distance between two points (x1, y1) and (x2, y2):
`distance = sqrt((x2 - x1)^2 + (y2 - y1)^2)`

A small difference in 'sepal length' (e.g., 1 cm) will contribute more to the distance than a significant difference in 'petal width' (e.g., 1 cm, which is a large portion of its range) simply because its values are larger.

**StandardScaler** transforms the data such that each feature has a mean of 0 and a standard deviation of 1. This process ensures that all features contribute equally to the distance calculations, preventing features with naturally larger values from dominating the distance metric. This is crucial for KNN to accurately identify true nearest neighbors based on the underlying patterns rather than feature scales.

## Task 2: Manual Implementation of KNN (Using Functions Only)

In [10]:
# Function to compute Euclidean distance
def euclidean_distance(x1, x2):
    """Compute Euclidean distance between two points"""
    return np.sqrt(np.sum((x1 - x2) ** 2))

# Function to predict class for a single sample
def predict_single(x, X_train, y_train, k):
    """Predict class for a single sample"""
    # Calculate distances to all training points
    distances = []
    for i in range(len(X_train)):
        dist = euclidean_distance(x, X_train[i])
        # Access y_train by integer position using .iloc
        distances.append((dist, y_train.iloc[i]))

    # Sort by distance and get k nearest neighbors
    distances.sort(key=lambda x: x[0])
    k_nearest = distances[:k]

    # Get labels of k nearest neighbors
    k_labels = [label for dist, label in k_nearest]

    # Majority vote
    # Use Counter to get the most common element
    most_common = Counter(k_labels).most_common(1)[0][0]
    return most_common

# Function to predict classes for multiple samples
def predict_all(X_test, X_train, y_train, k):
    """Predict classes for multiple samples"""
    predictions = []
    for x in X_test:
        predictions.append(predict_single(x, X_train, y_train, k))
    return np.array(predictions)

# Function to calculate accuracy manually
def accuracy_score_manual(y_true, y_pred):
    """Calculate accuracy manually"""
    return np.mean(y_true == y_pred)

# Define k_values to test
k_values = [1, 3, 5, int(np.sqrt(len(X_train)))]

print("Manual KNN Implementation Results:")
print("-" * 50)

manual_results = []

for k in k_values:
    # Predict on training data (just for completeness, usually not evaluated this way)
    y_train_pred = predict_all(X_train, X_train, y_train, k)
    train_accuracy = accuracy_score_manual(y_train, y_train_pred)

    # Predict on test data
    y_test_pred = predict_all(X_test, X_train, y_train, k)
    test_accuracy = accuracy_score_manual(y_test, y_test_pred)

    manual_results.append({
        'K': k,
        'Training Accuracy': train_accuracy,
        'Test Accuracy': test_accuracy
    })

    print(f"\nK = {k} {'(√N)' if k == int(np.sqrt(len(X_train))) else ''}:")
    print(f"Training Accuracy: {train_accuracy:.4f}")
    print(f"Test Accuracy: {test_accuracy:.4f}")

Manual KNN Implementation Results:
--------------------------------------------------

K = 1 :
Training Accuracy: 1.0000
Test Accuracy: 0.9333

K = 3 :
Training Accuracy: 0.9714
Test Accuracy: 0.9111

K = 5 :
Training Accuracy: 0.9810
Test Accuracy: 0.9111

K = 10 (√N):
Training Accuracy: 0.9810
Test Accuracy: 0.9556


### Observations from Manual KNN Implementation:

*   **K=1:** With K=1, the model is highly sensitive to noise in the training data, often leading to overfitting. The training accuracy is usually 100% because each training point is its own nearest neighbor. Test accuracy might be lower.

*   **K=3 and K=5:** As K increases from 1 to 3 or 5, the model becomes smoother and less sensitive to individual noisy points. This generally leads to better generalization and often a higher test accuracy, reducing overfitting.

*   **K=√N (Heuristic Selection):** This heuristic aims to find a balance between bias and variance. For our dataset, N is the number of training samples. This value of K often provides a good trade-off, leading to robust performance.

In general, increasing K helps to smooth out the decision boundaries and makes the model more robust to outliers, potentially reducing variance and improving test accuracy up to a point. If K becomes too large, the model might include neighbors from other classes, leading to underfitting and lower accuracy.

## Task 3: KNN using Scikit-learn

### Scikit-learn KNN Implementation and Evaluation

In [11]:
# Initialize lists to store results
sklearn_results = []

for k in k_values:
    # Create KNN classifier
    knn = KNeighborsClassifier(n_neighbors=k)

    # Train the model
    knn.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred = knn.predict(X_test)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)

    # For multiclass, precision, recall, f1-score need 'average' parameter
    # 'weighted' accounts for class imbalance
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    # ROC-AUC (One-vs-Rest for multiclass)
    # Get probabilities for ROC-AUC
    y_prob = knn.predict_proba(X_test)
    # Compute ROC AUC score for each class and average
    roc_auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')

    # Store results
    sklearn_results.append({
        'K': k,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Confusion Matrix': cm,
        'ROC-AUC': roc_auc
    })

    print(f"\nScikit-learn KNN with K = {k}:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print("Confusion Matrix:")
    print(cm)
    print(f"ROC-AUC: {roc_auc:.4f}")

# Display a summary table of scikit-learn results
print("\n" + "="*50)
print("Summary of Scikit-learn KNN Results:")
print("="*50)
for result in sklearn_results:
    print(f"K = {result['K']}: Accuracy={result['Accuracy']:.4f}, Precision={result['Precision']:.4f}, Recall={result['Recall']:.4f}, F1-Score={result['F1-Score']:.4f}, ROC-AUC={result['ROC-AUC']:.4f}")


Scikit-learn KNN with K = 1:
Accuracy: 0.9333
Precision: 0.9444
Recall: 0.9333
F1-Score: 0.9327
Confusion Matrix:
[[15  0  0]
 [ 0 15  0]
 [ 0  3 12]]
ROC-AUC: 0.9500

Scikit-learn KNN with K = 3:
Accuracy: 0.9111
Precision: 0.9298
Recall: 0.9111
F1-Score: 0.9095
Confusion Matrix:
[[15  0  0]
 [ 0 15  0]
 [ 0  4 11]]
ROC-AUC: 0.9607

Scikit-learn KNN with K = 5:
Accuracy: 0.9111
Precision: 0.9298
Recall: 0.9111
F1-Score: 0.9095
Confusion Matrix:
[[15  0  0]
 [ 0 15  0]
 [ 0  4 11]]
ROC-AUC: 0.9881

Scikit-learn KNN with K = 10:
Accuracy: 0.9333
Precision: 0.9444
Recall: 0.9333
F1-Score: 0.9327
Confusion Matrix:
[[15  0  0]
 [ 0 15  0]
 [ 0  3 12]]
ROC-AUC: 0.9948

Summary of Scikit-learn KNN Results:
K = 1: Accuracy=0.9333, Precision=0.9444, Recall=0.9333, F1-Score=0.9327, ROC-AUC=0.9500
K = 3: Accuracy=0.9111, Precision=0.9298, Recall=0.9111, F1-Score=0.9095, ROC-AUC=0.9607
K = 5: Accuracy=0.9111, Precision=0.9298, Recall=0.9111, F1-Score=0.9095, ROC-AUC=0.9881
K = 10: Accuracy=0.933

### Comparison and Interpretation of Metrics

**Comparison of Manual vs. Scikit-learn:**

Generally, the accuracy scores for the manual implementation and Scikit-learn's `KNeighborsClassifier` should be very similar, if not identical, given that both are likely using Euclidean distance by default and the same data split. Any minor differences might stem from floating-point precision or specific internal optimizations.

**Interpretation of Evaluation Metrics:**

*   **Accuracy:** This is the most straightforward metric, representing the proportion of correctly classified instances out of the total. For the Iris dataset, which is relatively balanced across classes, accuracy is a good general indicator of performance.

*   **Precision:** Precision measures the proportion of positive identifications that were actually correct. For each class, it tells us how many of the items predicted as that class truly belong to that class. High precision means fewer false positives.

*   **Recall (Sensitivity):** Recall measures the proportion of actual positives that were identified correctly. For each class, it tells us how many of the items that truly belong to that class were correctly predicted. High recall means fewer false negatives.

*   **F1-Score:** The F1-Score is the harmonic mean of precision and recall. It provides a single metric that balances both concerns. It's particularly useful when you need to balance precision and recall, especially in cases of uneven class distribution.

*   **Confusion Matrix:** This is a table that allows visualization of the performance of an algorithm. Each row of the matrix represents the instances in an actual class, while each column represents the instances in a predicted class. It helps to see where the model is making errors (e.g., misclassifying one flower species as another).

*   **ROC-AUC Score (One-vs-Rest for multiclass):** The Receiver Operating Characteristic (ROC) curve plots the True Positive Rate (Recall) against the False Positive Rate at various threshold settings. The Area Under the Curve (AUC) provides an aggregate measure of performance across all possible classification thresholds. For multiclass problems, the 'One-vs-Rest' (OvR) approach calculates the AUC for each class against all other classes, and then averages them. A higher ROC-AUC indicates a better ability of the model to distinguish between classes.

**Which metric is most informative for this problem?**

For the Iris dataset, which is a relatively balanced multiclass classification problem and not highly imbalanced, **Accuracy** and **F1-Score** are generally very informative.

*   **Accuracy** gives a quick overall understanding of correct classifications.
*   **F1-Score** is good because it considers both precision and recall, providing a balanced view, which is important if misclassifying any particular species has consequences.
*   The **Confusion Matrix** is also crucial as it provides a detailed breakdown of correct and incorrect predictions for each class, helping to identify specific types of misclassifications.